# Explanation of Replication Package
In this chapter, we will explain what we downloaded in our previous chapter.

We use the dataset metadata not only to list the files, but also to retrieve short explanations for them. This helps us understand what each downloaded file contains before opening it manually.

In [2]:
import json

# Load metadata from local file
with open("dataset_metadata.json", "r") as f:
    dataset_metadata = json.load(f)
files = dataset_metadata["data"]["latestVersion"]["files"]

for f in files:
    name = f["dataFile"]["filename"]
    size = f["dataFile"]["filesize"]
    restricted = f["restricted"]
    
    # Try to get description
    description = f.get("description")
    
    # Fallback if description is missing
    if not description:
        description = "No description available"
    
    print(f"File: {name}")
    print(f"Size: {size} bytes")
    print(f"Access: {'open' if not restricted else 'restricted'}")
    print(f"Explanation: {description}")
    print("-" * 40)

File: Assessment and Monitoring of Local Climate Regulation in Cities by Green Infrastructure—A National Ecosystem Service Indicator for Ge.pdf
Size: 11200912 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Climate regulation in cities as an ecosystem service_German.pdf
Size: 16359728 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Cooling_Capacity_2018_buffered.gdb.zip
Size: 229243224 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Documentation.md
Size: 7247 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Figure_1_Urban_green_Infrastructure_Syrbe_KLu-04.png
Size: 471283 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Map_1_Climate_regulation_Air_Photo.jpg
Size: 973971 bytes
Access: open
Explanation: No descri

This cell reads the file-level metadata and prints a short explanation for each file. It first looks for a file description, then for categories or folder information, and clearly shows whether the file is openly accessible or restricted.

Now, let's structure the metadata in a table for easier understanding.

In [3]:
import pandas as pd

rows = []

for f in files:
    rows.append({
        "filename": f["dataFile"]["filename"],
        "size_bytes": f["dataFile"]["filesize"],
        "access": "open" if not f["restricted"] else "restricted",
        "explanation": f.get("description", "No description available")
    })

df = pd.DataFrame(rows)
df

,filename,size_bytes,access,explanation
0,Assessment and Monitoring of Local Climate Reg...,11200912,open,No description available
1,Climate regulation in cities as an ecosystem s...,16359728,open,No description available
2,Cooling_Capacity_2018_buffered.gdb.zip,229243224,open,No description available
3,Documentation.md,7247,open,No description available
4,Figure_1_Urban_green_Infrastructure_Syrbe_KLu-...,471283,open,
5,Map_1_Climate_regulation_Air_Photo.jpg,973971,open,
6,Map_2_Climate_regulation_Tree_Cover.jpg,1071537,open,
7,Map_3_Climate_regulation_Population.jpg,790493,open,
8,Map_4_Climate_regulation_Cooling_Capacity.jpg,1145944,open,
9,Map_5_Cities_cooling_capacity.jpg,1697718,open,


This version presents the same information in a table format using pandas:

- Each row represents one file
- Columns show name, size, access, and explanation
- Easier to read and compare files

The downloaded dataset presents a national indicator of local climate regulation provided by urban green infrastructure (UGI) across 165 German cities. It quantifies the physical cooling capacity of urban green spaces and the proportion of the population benefiting from these services using geospatial methods. (2025-06-16) 

# Understanding the Restricted Geopackage file

In the next code block, we try to read the GeoPackage & inspect layers

- **GPKG** — path to the GeoPackage file  
- **`gpd.list_layers(GPKG)`** — list all layers in the GeoPackage (including spatial & non-spatial)  
- **Filter spatial layers**: `layers.geometry_type.notna()` picks layers with geometry  
- **Select first spatial layer**: `first_layer = … .iloc[0]`  
- **`gpd.read_file(...)`** with `layer=first_layer` — loads that layer into a GeoDataFrame  
- **`.head()`** — show first few rows  

**Assumptions & warnings:**
- The file path must be correct and accessible  
- There must be at least one spatial layer  
- The “first” spatial layer might not be the one you want — sometimes pick by name instead  


In [12]:
import pandas as pd
import geopandas as gpd

GPKG = "../../00_data/downloaded_data/climate_regulation_in_cities.gpkg"

#GPKG = "data/raw/climate_regulation_in_cities.gpkg"

# List all layers (spatial & non-spatial)
layers = gpd.list_layers(GPKG)
print("Available layers:")
print(layers)

# Filter to spatial layers and pick the first one by name
first_layer = layers.loc[layers.geometry_type.notna(), "name"].iloc[0]
print("Reading layer:", first_layer)

# Read the selected spatial layer into a GeoDataFrame
cities = gpd.read_file(GPKG, layer=first_layer)
print(cities.head())


Available layers:
                                    name geometry_type
0  climate_regulation_cities__refactored  MultiPolygon
Reading layer: climate_regulation_cities__refactored
        AGS       GEN   NUTS     Shape_Leng    Shape_Area  \
0  09563000     Fürth  DE253   49209.551437  6.334375e+07   
1  09162000   München  DE212  118084.194383  3.108342e+08   
2  09461000   Bamberg  DE241   44492.667416  5.464215e+07   
3  09262000    Passau  DE222   61360.769487  6.956591e+07   
4  09564000  Nürnberg  DE254  153997.952475  1.866370e+08   

   Pop_Benefit_Percent                                           geometry  
0            49.550707  MULTIPOLYGON (((4392577.758 2936451.016, 43926...  
1            70.270877  MULTIPOLYGON (((4432133.707 2774479.63, 443212...  
2            77.048748  MULTIPOLYGON (((4388203.406 2978639.586, 43882...  
3            78.069768  MULTIPOLYGON (((4570163.06 2838978.815, 457018...  
4            46.746620  MULTIPOLYGON (((4401927.814 2917518.273, 44019..

If you do not have access to the data, you may encounter errors. However, you still can get an idea of the data structure below.

# Meanings of each field are as follows 
- `AGS`: Official municipality key
- `GEN`: Municipality name
- `NUTS`: NUTS region code
- `shape_area`: Polygon area (m²)
- `shape_leng`: Polygon perimeter (m)
- `Pop_Benefit_Percent`: Percentage of inhabitants favored by the cooling effect of urban green infrastructure


In [13]:
cities.info()
cols = [c for c in cities.columns if c != cities.geometry.name]
summary = cities[cols].describe(include="all")
summary

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   AGS                  165 non-null    object  
 1   GEN                  165 non-null    object  
 2   NUTS                 165 non-null    object  
 3   Shape_Leng           165 non-null    float64 
 4   Shape_Area           165 non-null    float64 
 5   Pop_Benefit_Percent  165 non-null    float64 
 6   geometry             165 non-null    geometry
dtypes: float64(3), geometry(1), object(3)
memory usage: 9.2+ KB


,AGS,GEN,NUTS,Shape_Leng,Shape_Area,Pop_Benefit_Percent
count,165,165,165,165.000000,1.650000e+02,165.000000
unique,165,165,137,NaN,NaN,NaN
top,09563000,Fürth,DEA36,NaN,NaN,NaN
freq,1,1,6,NaN,NaN,NaN
mean,NaN,NaN,NaN,73252.695637,1.261561e+08,75.851682
std,NaN,NaN,NaN,31028.459267,1.044051e+08,10.090394
min,NaN,NaN,NaN,27046.322289,2.594714e+07,46.746620
25%,NaN,NaN,NaN,53928.592357,6.977318e+07,69.224536
50%,NaN,NaN,NaN,65442.570260,9.850075e+07,76.193042
75%,NaN,NaN,NaN,86071.639745,1.529937e+08,83.391564


# What this code does
Displays a quick overview of the dataset:
- number of rows and columns
- column names
- data types (e.g., numeric, string, geometry)
- missing values
This helps you understand the dataset's structure.

# Overall Takeaway

In this chapter, we learned how to understand the metadata. In the next chapter, we will have some basic visualisation for better understanding. 